# Visual Product Search — Full Pipeline
### Configs A / B / C · Fused Embeddings · BLIP-ITM Re-ranking · Multi-seed Ablation

In [ ]:
!pip install -q transformers ftfy accelerate faiss-cpu

## Setup

In [ ]:
import os, json, random, gc, pickle, numpy as np, pandas as pd
from PIL import Image
import torch, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from transformers import CLIPModel, CLIPProcessor
import faiss, warnings; warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

BASE     = '/kaggle/input/datasets/sasank93/cropped-images-vr-final-project'
IMG_DIR  = os.path.join(BASE, 'cropped_img/cropped_img')
CAPTIONS = os.path.join(BASE, 'captions.json')
OUT      = '/kaggle/working'
os.makedirs(OUT, exist_ok=True)

SEEDS     = [42, 123, 456]
CLIP_NAME = 'openai/clip-vit-base-patch32'
BLIP_NAME = 'Salesforce/blip-itm-base-coco'
ALPHAS    = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
K_LIST    = [5, 10, 15]

print('IMG_DIR:', os.path.isdir(IMG_DIR))
print('CAPTIONS:', os.path.isfile(CAPTIONS))

## 1. Load Data

In [ ]:
with open(CAPTIONS) as f:
    captions = json.load(f)

records = []
for p, c in captions.items():
    iid = next((x for x in p.split('/') if x.startswith('id_')), None)
    fp = os.path.join(IMG_DIR, p)
    if iid and os.path.isfile(fp):
        records.append({'image_name': p, 'full_path': fp, 'item_id': iid, 'caption': c})

df = pd.DataFrame(records)
print(f'Images: {len(df)} | Items: {df["item_id"].nunique()}')

# Check images per item distribution
counts = df.groupby('item_id').size()
print(f'Images per item — min: {counts.min()}, max: {counts.max()}, mean: {counts.mean():.1f}')
print(f'Items with >=2 images: {(counts >= 2).sum()}')
df.head(2)

## 2. Correct Train / Query / Gallery Split
> Key: eval items must have images in BOTH query and gallery.

In [ ]:
def make_split(df, seed, train_frac=0.7):
    """Split so that query and gallery share the same item_ids.
    - Train: 70% of item_ids (all images)
    - Eval:  30% of item_ids, each item split into 1 query + rest gallery
    Items with only 1 image go to train (can't split).
    """
    rng = random.Random(seed)

    counts = df.groupby('item_id').size()
    # Only items with >=2 images can be used for eval
    multi_items = counts[counts >= 2].index.tolist()
    single_items = counts[counts == 1].index.tolist()
    rng.shuffle(multi_items)

    n_train = int(train_frac * len(multi_items))
    train_item_ids = set(multi_items[:n_train]) | set(single_items)
    eval_item_ids  = multi_items[n_train:]

    train_df = df[df['item_id'].isin(train_item_ids)].reset_index(drop=True)

    # For eval items: 1 random image -> query, rest -> gallery
    query_rows, gallery_rows = [], []
    for iid in eval_item_ids:
        item_imgs = df[df['item_id'] == iid].sample(frac=1, random_state=seed).reset_index(drop=True)
        query_rows.append(item_imgs.iloc[0])
        for j in range(1, len(item_imgs)):
            gallery_rows.append(item_imgs.iloc[j])

    query_df   = pd.DataFrame(query_rows).reset_index(drop=True)
    gallery_df = pd.DataFrame(gallery_rows).reset_index(drop=True)

    # Verify overlap
    q_ids = set(query_df['item_id'])
    g_ids = set(gallery_df['item_id'])
    overlap = q_ids & g_ids
    print(f'  Train: {len(train_df)} imgs | Query: {len(query_df)} imgs | Gallery: {len(gallery_df)} imgs')
    print(f'  Eval items: {len(eval_item_ids)} | Q-G overlap: {len(overlap)} (should = {len(eval_item_ids)})')

    return train_df, query_df, gallery_df

# Quick test
_, _q, _g = make_split(df, 42)
print(f'  Sample query item: {_q.iloc[0]["item_id"]}')
print(f'  Same item in gallery: {(_g["item_id"] == _q.iloc[0]["item_id"]).sum()} images')

## 3. All Helper Functions

In [ ]:
class PairDatasetV2(Dataset):
    def __init__(self, df, processor, max_pairs=40000):
        self.processor = processor
        groups = df.groupby('item_id')['full_path'].apply(list)
        self.pairs = []
        for iid, paths in groups.items():
            if len(paths) >= 2:
                target = min(len(paths)*(len(paths)-1)//2, 8)
                seen, att = set(), 0
                while len(seen) < target and att < 40:
                    a, b = random.sample(paths, 2)
                    seen.add(tuple(sorted([a, b])))
                    att += 1
                self.pairs.extend(seen)
            if len(self.pairs) >= max_pairs: break
        random.shuffle(self.pairs)
        print(f'Pairs: {len(self.pairs)}')
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        a, b = self.pairs[idx]
        ai = Image.open(a).convert('RGB')
        bi = Image.open(b).convert('RGB')
        av = self.processor(images=ai, return_tensors='pt')['pixel_values'].squeeze(0)
        bv = self.processor(images=bi, return_tensors='pt')['pixel_values'].squeeze(0)
        return av, bv

def infonce_loss(a, b, temp=0.07):
    a, b = F.normalize(a, dim=-1), F.normalize(b, dim=-1)
    logits = (a @ b.T) / temp
    labels = torch.arange(len(a), device=a.device)
    return (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2

@torch.no_grad()
def extract_visual(df, model, proc, bs=128, tag=''):
    model.eval(); embs = []; paths = df['full_path'].tolist()
    for s in range(0, len(paths), bs):
        imgs = []
        for p in paths[s:s+bs]:
            try: imgs.append(Image.open(p).convert('RGB'))
            except: imgs.append(Image.new('RGB', (224,224)))
        inp = proc(images=imgs, return_tensors='pt', padding=True).to(DEVICE)
        out = model.vision_model(pixel_values=inp['pixel_values'])
        e = F.normalize(model.visual_projection(out.pooler_output), dim=-1)
        embs.append(e.cpu().numpy())
        if (s//bs) % 20 == 0: print(f'  {tag} {s}/{len(paths)}')
    return np.vstack(embs)

@torch.no_grad()
def extract_text(df, model, proc, bs=256, tag=''):
    model.eval(); embs = []; caps = df['caption'].tolist()
    for s in range(0, len(caps), bs):
        inp = proc(text=caps[s:s+bs], return_tensors='pt', padding=True, truncation=True, max_length=77).to(DEVICE)
        out = model.text_model(input_ids=inp['input_ids'], attention_mask=inp['attention_mask'])
        e = F.normalize(model.text_projection(out.pooler_output), dim=-1)
        embs.append(e.cpu().numpy())
    return np.vstack(embs)

def fuse(vis, txt, alpha):
    f = alpha * vis + (1-alpha) * txt
    return f / (np.linalg.norm(f, axis=1, keepdims=True) + 1e-8)

def build_index(embs):
    idx = faiss.IndexHNSWFlat(embs.shape[1], 32)
    idx.hnsw.efConstruction = 200
    idx.add(embs.astype('float32'))
    return idx

def do_search(index, q, k):
    index.hnsw.efSearch = max(100, k*4)
    _, I = index.search(q.astype('float32'), k=k)
    return I

def calc_metrics(q_ids, g_ids, I, ks):
    ret = g_ids[I]; rel = (ret == q_ids[:, None]).astype(float); res = {}
    for k in ks:
        r = rel[:, :k]; rk = np.arange(1, k+1)
        res[f'Recall@{k}'] = float(r.any(1).mean())
        dcg = (r / np.log2(rk+1)).sum(1)
        # iDCG: count how many relevant items exist per query in gallery
        n_rel = rel.sum(1)  # total relevant per query
        ideal = np.zeros_like(dcg)
        for i in range(len(dcg)):
            nr = int(min(n_rel[i], k))
            if nr > 0: ideal[i] = sum(1/np.log2(j+2) for j in range(nr))
        ndcg = np.divide(dcg, ideal, out=np.zeros_like(dcg), where=ideal>0)
        res[f'NDCG@{k}'] = float(ndcg.mean())
        ch = r.cumsum(1)
        ap = (r * ch / rk).sum(1) / np.maximum(n_rel, 1).clip(max=k)
        res[f'mAP@{k}'] = float(ap.mean())
    return res

def tune_alpha(qv, qt, gv, gt, qi, gi, alphas):
    best_a, best_v = 0.5, -1
    for a in alphas:
        idx = build_index(fuse(gv, gt, a))
        I = do_search(idx, fuse(qv, qt, a), 10)
        v = calc_metrics(qi, gi, I, [10])['Recall@10']
        print(f'  a={a:.1f} -> R@10={v:.4f}')
        if v > best_v: best_v, best_a = v, a
    print(f'  Best: a={best_a} (R@10={best_v:.4f})')
    return best_a

print('All helpers ready.')

## 4. Training Function

In [ ]:
def train_clip(model, train_df, proc, max_ep=15, min_ep=6, lr=1e-5, patience=3):
    for p in model.parameters(): p.requires_grad = False
    tp = []
    for layer in model.vision_model.encoder.layers[-4:]:
        for p in layer.parameters(): p.requires_grad = True; tp.append(p)
    for p in model.vision_model.post_layernorm.parameters(): p.requires_grad = True; tp.append(p)
    print(f'  Trainable: {sum(x.numel() for x in tp):,}')

    opt = torch.optim.AdamW(tp, lr=lr, weight_decay=1e-4)
    sched = CosineAnnealingLR(opt, T_max=max_ep, eta_min=lr/10)
    best, stale = float('inf'), 0

    for ep in range(max_ep):
        ds = PairDatasetV2(train_df, proc, max_pairs=40000)
        loader = DataLoader(ds, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
        model.train(); total = 0
        for step, (a, b) in enumerate(loader):
            a, b = a.to(DEVICE), b.to(DEVICE)
            ae = model.visual_projection(model.vision_model(pixel_values=a).pooler_output)
            be = model.visual_projection(model.vision_model(pixel_values=b).pooler_output)
            loss = infonce_loss(ae, be)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(tp, 1.0)
            opt.step(); total += loss.item()
        avg = total / len(loader); sched.step()
        print(f'  Ep {ep+1:02d} | loss={avg:.4f} | lr={opt.param_groups[0]["lr"]:.1e}')
        if ep >= min_ep - 1:
            if best - avg < 0.003: stale += 1
            else: stale = 0
            if avg < best: best = avg
            if stale >= patience: print(f'  Early stop ep {ep+1}'); break
        else:
            if avg < best: best = avg
    return model

print('Training ready.')

## 5. BLIP-ITM Re-ranking

In [ ]:
def rerank_blip(query_df, gallery_df, q_embs, g_embs, k_list, top_n=50, tag=''):
    from transformers import BlipProcessor, BlipForImageTextRetrieval
    idx = build_index(g_embs)
    I_raw = do_search(idx, q_embs, top_n)
    g_arr, q_arr = gallery_df['item_id'].values, query_df['item_id'].values
    m_before = calc_metrics(q_arr, g_arr, I_raw, k_list)

    print(f'  [{tag}] Loading BLIP-ITM...')
    bp = BlipProcessor.from_pretrained(BLIP_NAME)
    bm = BlipForImageTextRetrieval.from_pretrained(BLIP_NAME).to(DEVICE)
    bm.eval()

    I_re = np.zeros_like(I_raw)
    for qi in range(len(query_df)):
        ci = I_raw[qi]
        caps = gallery_df.iloc[ci]['caption'].tolist()
        qimg = Image.open(query_df.iloc[qi]['full_path']).convert('RGB')
        with torch.no_grad():
            inp = bp(images=[qimg]*len(caps), text=caps,
                     return_tensors='pt', padding=True, truncation=True, max_length=40).to(DEVICE)
            sc = torch.softmax(bm(**inp, use_itm_head=True).itm_score, dim=1)[:,1].cpu().numpy()
        I_re[qi] = ci[np.argsort(-sc)]
        if qi % 500 == 0: print(f'  [{tag}] {qi}/{len(query_df)}')

    del bm, bp; torch.cuda.empty_cache(); gc.collect()
    print(f'  [{tag}] BLIP unloaded.')
    return m_before, calc_metrics(q_arr, g_arr, I_re, k_list)

print('Re-ranking ready.')

## 6. Main Loop — 3 Seeds

In [ ]:
rpath = f'{OUT}/all_results.pkl'
if os.path.exists(rpath):
    with open(rpath, 'rb') as f: all_results = pickle.load(f)
    print('Resumed:', list(all_results.keys()))
else:
    all_results = {}

for seed in SEEDS:
    if seed in all_results and 'C_reranked' in all_results[seed]:
        print(f'Seed {seed} done, skip.'); continue

    print(f'\n{"="*60}\nSEED {seed}\n{"="*60}')
    try:
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        if torch.cuda.is_available(): torch.cuda.manual_seed(seed)

        train_df, query_df, gallery_df = make_split(df, seed)
        qi, gi = query_df['item_id'].values, gallery_df['item_id'].values

        clip = CLIPModel.from_pretrained(CLIP_NAME).to(DEVICE)
        proc = CLIPProcessor.from_pretrained(CLIP_NAME)
        clip.eval()

        # CONFIG A
        print('\n[A] Embeddings...')
        gvA = extract_visual(gallery_df, clip, proc, tag='gA')
        qvA = extract_visual(query_df, clip, proc, tag='qA')
        gt  = extract_text(gallery_df, clip, proc, tag='gT')
        qt  = extract_text(query_df, clip, proc, tag='qT')
        mA = calc_metrics(qi, gi, do_search(build_index(gvA), qvA, max(K_LIST)), K_LIST)
        print('[A]', mA)

        # CONFIG B
        print('\n[B] Alpha tuning...')
        aB = tune_alpha(qvA, qt, gvA, gt, qi, gi, ALPHAS)
        gfB, qfB = fuse(gvA, gt, aB), fuse(qvA, qt, aB)
        print('[B] BLIP re-ranking...')
        mB_pre, mB_post = rerank_blip(query_df, gallery_df, qfB, gfB, K_LIST, tag='B')
        print('[B] pre:', mB_pre)
        print('[B] post:', mB_post)

        # TRAIN
        print('\n[Train] Fine-tuning...')
        clip = train_clip(clip, train_df, proc)
        clip.save_pretrained(f'{OUT}/clip_ft_s{seed}')

        # CONFIG C
        print('\n[C] Embeddings (fine-tuned)...')
        gvC = extract_visual(gallery_df, clip, proc, tag='gC')
        qvC = extract_visual(query_df, clip, proc, tag='qC')
        print('[C] Alpha tuning...')
        aC = tune_alpha(qvC, qt, gvC, gt, qi, gi, ALPHAS)
        gfC, qfC = fuse(gvC, gt, aC), fuse(qvC, qt, aC)
        print('[C] BLIP re-ranking...')
        mC_pre, mC_post = rerank_blip(query_df, gallery_df, qfC, gfC, K_LIST, tag='C')
        print('[C] pre:', mC_pre)
        print('[C] post:', mC_post)

        # Save
        for nm, ar in [('gvA',gvA),('qvA',qvA),('gvC',gvC),('qvC',qvC),('gt',gt),('qt',qt)]:
            np.save(f'{OUT}/s{seed}_{nm}.npy', ar)
        gallery_df.to_csv(f'{OUT}/s{seed}_gal.csv', index=False)
        query_df.to_csv(f'{OUT}/s{seed}_qry.csv', index=False)

        all_results[seed] = {
            'A_no_rerank': mA,
            'B_no_rerank': mB_pre, 'B_reranked': mB_post,
            'C_no_rerank': mC_pre, 'C_reranked': mC_post,
            'alpha_B': aB, 'alpha_C': aC
        }
        del clip, proc; torch.cuda.empty_cache(); gc.collect()
        print(f'\nSeed {seed} DONE.')

    except Exception as e:
        print(f'\n*** ERROR seed {seed}: {e}')
        import traceback; traceback.print_exc()

    with open(rpath, 'wb') as f: pickle.dump(all_results, f)
    print(f'Saved ({len(all_results)} seeds).')

print('\n=== ALL COMPLETE ===')
print('Seeds:', list(all_results.keys()))

## 7. Results — Mean ± Std

In [ ]:
with open(f'{OUT}/all_results.pkl', 'rb') as f:
    all_results = pickle.load(f)

cfg_names = {
    'A_no_rerank': 'A: Frozen CLIP, visual only (a=1)',
    'B_no_rerank': 'B: Frozen CLIP + fused (no rerank)',
    'B_reranked':  'B+: Frozen CLIP + fused + BLIP rerank',
    'C_no_rerank': 'C: Tuned CLIP + fused (no rerank)',
    'C_reranked':  'C+: Tuned CLIP + fused + BLIP rerank',
}
mks = [f'{m}@{k}' for m in ['Recall','NDCG','mAP'] for k in K_LIST]

rows = []
for cfg, label in cfg_names.items():
    row = {'Config': label}
    for mk in mks:
        vals = [all_results[s][cfg][mk] for s in all_results if cfg in all_results[s]]
        row[mk] = f'{np.mean(vals):.4f} +/- {np.std(vals):.4f}' if vals else 'N/A'
    rows.append(row)

rdf = pd.DataFrame(rows).set_index('Config')
rdf.to_csv(f'{OUT}/ablation_results.csv')
print(rdf.to_string())

print('\nAlpha values:')
for s in all_results:
    r = all_results[s]
    print(f"  Seed {s}: aB={r.get('alpha_B','?')}, aC={r.get('alpha_C','?')}")